In [1]:
from dataclasses import dataclass
from pathlib import Path
import io
import re

import zstandard as zstd

In [2]:
@dataclass
class PgnGame:
  game_index: int
  tags: dict
  movetext: str
  raw_pgn: str


class PgnZstParser:
  def __init__(self, path):
    self.path = Path(path)

  def parse_first_n(self, n_games):
    games = []

    for game in self.iter_games():
      games.append(game)

      if len(games) >= n_games:
        break

    return games

  def iter_games(self):
    game_lines = []
    game_index = 0
    seen_movetext = False

    with open(self.path, "rb") as file:
      dctx = zstd.ZstdDecompressor()

      with dctx.stream_reader(file) as byte_stream:
        text_stream = io.TextIOWrapper(
          byte_stream,
          encoding="utf-8",
          errors="replace",
        )

        for line in text_stream:
          line = line.rstrip("\n")

          if self._is_new_game_start(
            line,
            game_lines,
            seen_movetext,
          ):
            yield self._build_game(game_index, game_lines)
            game_index += 1
            game_lines = []
            seen_movetext = False

          game_lines.append(line)

          if self._is_movetext_line(line):
            seen_movetext = True

        if game_lines:
          yield self._build_game(game_index, game_lines)

  def _is_new_game_start(self, line, game_lines, seen_movetext):
    if not game_lines:
      return False

    if not seen_movetext:
      return False

    return line.startswith("[Event ")

  def _is_movetext_line(self, line):
    if not line.strip():
      return False

    if line.startswith("[") and line.endswith("]"):
      return False

    return True

  def _build_game(self, game_index, game_lines):
    raw_pgn = "\n".join(game_lines).strip()
    tags = {}
    movetext_lines = []
    in_movetext = False

    for line in game_lines:
      if self._is_tag_line(line) and not in_movetext:
        key, value = self._parse_tag_line(line)
        tags[key] = value
        continue

      if line.strip():
        in_movetext = True

      if in_movetext:
        movetext_lines.append(line)

    movetext = "\n".join(movetext_lines).strip()

    return PgnGame(
      game_index=game_index,
      tags=tags,
      movetext=movetext,
      raw_pgn=raw_pgn,
    )

  def _is_tag_line(self, line):
    return bool(re.match(r'^\[[A-Za-z0-9_]+ ".*"\]$', line))

  def _parse_tag_line(self, line):
    match = re.match(r'^\[([A-Za-z0-9_]+) "(.*)"\]$', line)

    if match is None:
      raise ValueError(f"Could not parse tag line: {line}")

    return match.group(1), match.group(2)

In [3]:
pgn_path = "../data/raw/lichess_db_standard_rated_2017-05.pgn.zst"

parser = PgnZstParser(pgn_path)
games = parser.parse_first_n(5)

In [4]:
games[0].tags

{'Event': 'Rated Bullet game',
 'Site': 'https://lichess.org/ObT3MGJ6',
 'White': 'marshall91',
 'Black': 'Nechemevich',
 'Result': '0-1',
 'UTCDate': '2017.04.30',
 'UTCTime': '22:00:00',
 'WhiteElo': '1926',
 'BlackElo': '1969',
 'WhiteRatingDiff': '-10',
 'BlackRatingDiff': '+9',
 'ECO': 'B01',
 'Opening': 'Scandinavian Defense: Modern Variation #2',
 'TimeControl': '60+0',
 'Termination': 'Time forfeit'}

In [5]:
print(games[0].movetext)

1. e4 { [%clk 0:01:00] } d5 { [%clk 0:01:00] } 2. exd5 { [%clk 0:00:59] } Nf6 { [%clk 0:01:00] } 3. d3 { [%clk 0:00:57] } Qxd5 { [%clk 0:00:58] } 4. Nc3 { [%clk 0:00:56] } Qf5 { [%clk 0:00:56] } 5. Be2 { [%clk 0:00:56] } Bd7 { [%clk 0:00:54] } 6. g4 { [%clk 0:00:56] } Qe6 { [%clk 0:00:52] } 7. g5 { [%clk 0:00:55] } Nd5 { [%clk 0:00:51] } 8. Ne4 { [%clk 0:00:53] } Bc6 { [%clk 0:00:50] } 9. Bg4 { [%clk 0:00:52] } Qe5 { [%clk 0:00:48] } 10. f4 { [%clk 0:00:51] } Qd4 { [%clk 0:00:47] } 11. Nf3 { [%clk 0:00:49] } Qb6 { [%clk 0:00:46] } 12. Qe2 { [%clk 0:00:47] } e6 { [%clk 0:00:44] } 13. Be3 { [%clk 0:00:45] } Qa6 { [%clk 0:00:43] } 14. O-O { [%clk 0:00:44] } Nd7 { [%clk 0:00:42] } 15. c4 { [%clk 0:00:42] } Ne7 { [%clk 0:00:41] } 16. f5 { [%clk 0:00:39] } Bxe4 { [%clk 0:00:39] } 17. dxe4 { [%clk 0:00:38] } exf5 { [%clk 0:00:39] } 18. exf5 { [%clk 0:00:37] } O-O-O { [%clk 0:00:39] } 19. f6 { [%clk 0:00:36] } gxf6 { [%clk 0:00:38] } 20. gxf6 { [%clk 0:00:35] } Nc6 { [%clk 0:00:35] } 21. Nd4 {

In [6]:
print(games[0].raw_pgn)

[Event "Rated Bullet game"]
[Site "https://lichess.org/ObT3MGJ6"]
[White "marshall91"]
[Black "Nechemevich"]
[Result "0-1"]
[UTCDate "2017.04.30"]
[UTCTime "22:00:00"]
[WhiteElo "1926"]
[BlackElo "1969"]
[WhiteRatingDiff "-10"]
[BlackRatingDiff "+9"]
[ECO "B01"]
[Opening "Scandinavian Defense: Modern Variation #2"]
[TimeControl "60+0"]
[Termination "Time forfeit"]

1. e4 { [%clk 0:01:00] } d5 { [%clk 0:01:00] } 2. exd5 { [%clk 0:00:59] } Nf6 { [%clk 0:01:00] } 3. d3 { [%clk 0:00:57] } Qxd5 { [%clk 0:00:58] } 4. Nc3 { [%clk 0:00:56] } Qf5 { [%clk 0:00:56] } 5. Be2 { [%clk 0:00:56] } Bd7 { [%clk 0:00:54] } 6. g4 { [%clk 0:00:56] } Qe6 { [%clk 0:00:52] } 7. g5 { [%clk 0:00:55] } Nd5 { [%clk 0:00:51] } 8. Ne4 { [%clk 0:00:53] } Bc6 { [%clk 0:00:50] } 9. Bg4 { [%clk 0:00:52] } Qe5 { [%clk 0:00:48] } 10. f4 { [%clk 0:00:51] } Qd4 { [%clk 0:00:47] } 11. Nf3 { [%clk 0:00:49] } Qb6 { [%clk 0:00:46] } 12. Qe2 { [%clk 0:00:47] } e6 { [%clk 0:00:44] } 13. Be3 { [%clk 0:00:45] } Qa6 { [%clk 0:00:43